In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## 2.1 基础清洗

### 2.1.1 编码识别与文件读取

依次读取 `data/raw/` 下的 4 个有效数据文件：
- `STK_LISTEDCOINFOANL.xlsx`
- `STK_LISTEDCOINFOCHG.xlsx`
- `常用变量查询（年度）.xlsx`
- `跨表查询_沪深京股票(年频).xlsx`

对每个文件自动检测编码（主要针对可能存在的 CSV 文件），若为 Excel 文件则直接读取。

In [5]:
# 定义数据文件路径
raw_path = Path("data/raw")

# 手动指定四个有效文件（排除描述文件 .txt）
data_files = {
    "company_info": "STK_LISTEDCOINFOANL.xlsx",
    "company_chg": "STK_LISTEDCOINFOCHG.xlsx",
    "annual_vars": "常用变量查询（年度）.xlsx",
    "cross_table": "跨表查询_沪深京股票(年频).xlsx",
}

# 读取所有文件
dfs = {}
for name, filename in data_files.items():
    file_path = raw_path / filename
    if not file_path.exists():
        print(f"⚠️ 文件不存在: {filename}")
        continue
    try:
        # 尝试用 openpyxl 读取 Excel
        dfs[name] = pd.read_excel(file_path)
        print(f"✔ {filename} 读取成功，形状: {dfs[name].shape}")
    except Exception as e:
        print(f"✘ {filename} 读取失败: {e}")

✔ STK_LISTEDCOINFOANL.xlsx 读取成功，形状: (64173, 40)
✔ STK_LISTEDCOINFOCHG.xlsx 读取成功，形状: (160278, 8)
✔ 常用变量查询（年度）.xlsx 读取成功，形状: (61458, 33)
✔ 跨表查询_沪深京股票(年频).xlsx 读取成功，形状: (81665, 32)


### 2.1.2 主键统一

- **股票代码**：统一处理为 6 位字符串（如 `'000001'`），新列名为 `code`。
- **会计年度**：从报表日期或会计期间字段提取年份，转为整数型，列名 `year`。
- 注意不同表中股票代码字段名不同：
  - `company_info`、`company_chg` 中为 `Symbol`
  - `annual_vars` 中为 `Stkcd`
  - `cross_table` 中为 `code`

In [6]:
# 统一股票代码处理函数
def standardize_code(series):
    # 转为字符串，去除空格，若少于6位前面补0
    s = series.astype(str).str.strip().str.replace("'", "").str.replace('"', '')
    return s.str.zfill(6)

# 处理每个 DataFrame
# 1. 公司基本信息表
df_info = dfs["company_info"].copy()
df_info["code"] = standardize_code(df_info["Symbol"])
# 这里没有 year 字段，后续合并时只提供截面信息

# 2. 公司变更表
df_chg = dfs["company_chg"].copy()
df_chg["code"] = standardize_code(df_chg["Symbol"])
# 也没有 year，只记录变更历史

# 3. 常用变量查询（年度）
df_av = dfs["annual_vars"].copy()
df_av["code"] = standardize_code(df_av["Stkcd"])
# accper 格式如 '2020-12-31'，提取年份
df_av["year"] = pd.to_datetime(df_av["accper"], errors='coerce').dt.year
# 删除无法解析年份的行
df_av = df_av.dropna(subset=["year"])
df_av["year"] = df_av["year"].astype(int)

# 4. 跨表查询（年频）
df_ct = dfs["cross_table"].copy()
df_ct["code"] = standardize_code(df_ct["code"].astype(str))
# EndDate 是日期
df_ct["year"] = pd.to_datetime(df_ct["EndDate"], errors='coerce').dt.year
df_ct = df_ct.dropna(subset=["year"])
df_ct["year"] = df_ct["year"].astype(int)

print("主键统一完成")
print(f"annual_vars: {df_av.shape}, cross_table: {df_ct.shape}")

主键统一完成
annual_vars: (61456, 35), cross_table: (81662, 33)


### 2.1.3 日期变量统一

将 `listingDate`、`EndDate`、`ImplementDate` 等日期列转换为 `datetime64` 格式，便于后续计算。

In [7]:
# 公司信息表：上市日期
if "LISTINGDATE" in df_info.columns:
    df_info["listing_date"] = pd.to_datetime(df_info["LISTINGDATE"], errors='coerce')

# 常用变量表：accper 已转为 datetime 在上一节，也可保留原 datetime
df_av["accper_dt"] = pd.to_datetime(df_av["accper"], errors='coerce')

# 跨表查询：EndDate 已处理
df_ct["EndDate_dt"] = pd.to_datetime(df_ct["EndDate"], errors='coerce')

# 公司变更表：公告日和实施日
if "AnnouncementDate" in df_chg.columns:
    df_chg["announcement_date"] = pd.to_datetime(df_chg["AnnouncementDate"], errors='coerce')
if "ImplementDate" in df_chg.columns:
    df_chg["implement_date"] = pd.to_datetime(df_chg["ImplementDate"], errors='coerce')

print("日期变量转换完成")

日期变量转换完成


### 2.1.4 年度样本筛选

根据作业要求，保留会计年度 **2000 年及以后** 的观测。如果某指标原始起始年份晚于 2000，以实际年份为准。

In [8]:
# 只保留 year >= 2000
df_av = df_av[df_av["year"] >= 2000].copy()
df_ct = df_ct[df_ct["year"] >= 2000].copy()

print(f"筛选后 annual_vars: {df_av.shape}, cross_table: {df_ct.shape}")

筛选后 annual_vars: (61456, 36), cross_table: (81662, 34)


### 2.1.5 code-year 重复检查

对年度表检查是否存在同一公司同一年度多条记录。若存在重复，按以下规则保留：
- 对于 `annual_vars`，保留第一条（一般不应重复）。
- 对于 `cross_table`，也是保留第一条。
同时输出重复数量，说明处理方式。

In [9]:
# 检查 annual_vars
dup_av = df_av.duplicated(subset=["code", "year"], keep=False)
if dup_av.any():
    print(f"annual_vars 存在 {dup_av.sum()} 条重复（code-year），保留第一条。")
    df_av = df_av.drop_duplicates(subset=["code", "year"], keep="first")
else:
    print("annual_vars 无 code-year 重复。")

# 检查 cross_table
dup_ct = df_ct.duplicated(subset=["code", "year"], keep=False)
if dup_ct.any():
    print(f"cross_table 存在 {dup_ct.sum()} 条重复（code-year），保留第一条。")
    df_ct = df_ct.drop_duplicates(subset=["code", "year"], keep="first")
else:
    print("cross_table 无 code-year 重复。")

annual_vars 无 code-year 重复。
cross_table 无 code-year 重复。


### 2.1.6 缺失值统计与说明

统计两张核心年度表中主要变量的缺失数量和比例，分析可能原因（如公司上市晚、数据未披露等）。

In [10]:
# 定义需要检查的核心变量（从 variable_dictionary 中选取重要的）
vars_av = ["Ysmvttl", "Yretwd", "Shrcr1", "Boardsize2", "IndDirector"]  # 示例，按需修改
vars_ct = ["FS_Combas-A001000000", "FS_Combas-A002000000", "FS_Combas-A003000000"]  # 总资产、总负债、所有者权益

print("===== 常用变量缺失情况 =====")
for v in vars_av:
    if v in df_av.columns:
        miss = df_av[v].isna().sum()
        print(f"{v} 缺失 {miss} 条，占比 {miss/len(df_av):.2%}")
    else:
        print(f"{v} 不在数据集中")

print("\n===== 跨表查询缺失情况 =====")
for v in vars_ct:
    if v in df_ct.columns:
        miss = df_ct[v].isna().sum()
        print(f"{v} 缺失 {miss} 条，占比 {miss/len(df_ct):.2%}")
    else:
        print(f"{v} 不在数据集中")

===== 常用变量缺失情况 =====
Ysmvttl 缺失 0 条，占比 0.00%
Yretwd 缺失 4650 条，占比 7.57%
Shrcr1 缺失 3708 条，占比 6.03%
Boardsize2 缺失 319 条，占比 0.52%
IndDirector 缺失 357 条，占比 0.58%

===== 跨表查询缺失情况 =====
FS_Combas-A001000000 缺失 26951 条，占比 33.00%
FS_Combas-A002000000 缺失 26951 条，占比 33.00%
FS_Combas-A003000000 缺失 26951 条，占比 33.00%


### 原因分析

**统计结果：**

| 变量 | 缺失数量 | 缺失比例 | 所属表 |
|------|----------|-----------|--------|
| Ysmvttl (总市值) | 0 | 0.00% | 常用变量查询 |
| Yretwd (年个股回报率) | 4,650 | 7.57% | 常用变量查询 |
| Shrcr1 (第一大股东持股) | 3,708 | 6.03% | 常用变量查询 |
| Boardsize2 (董事会规模) | 319 | 0.52% | 常用变量查询 |
| IndDirector (独立董事人数) | 357 | 0.58% | 常用变量查询 |
| A001000000 (总资产) | 26,951 | 33.00% | 跨表查询 |
| A002000000 (总负债) | 26,951 | 33.00% | 跨表查询 |
| A003000000 (所有者权益) | 26,951 | 33.00% | 跨表查询 |

**原因分析：**

1. **Yretwd (年个股回报率) 缺失 7.57%**  
   原因：部分公司-年度因长期停牌、刚上市不足一年或数据库未计算当年回报率，导致缺失。属于正常数据缺失，不构成系统性偏差。

2. **Shrcr1 (第一大股东持股比例) 缺失 6.03%**  
   原因：少数公司未按时披露年报，或股权结构数据因特殊原因未收录。缺失率较低，对整体分析影响有限。

3. **Boardsize2 / IndDirector (董事会规模/独立董事人数) 缺失 <1%**  
   原因：公司治理数据从 2003 年起才在 CSMAR 中系统性收录，2000-2002 年极少数公司有缺失，属于数据库覆盖范围限制，不会显著影响后续结果。

4. **跨表查询表三大财务指标 (总资产/总负债/所有者权益) 同步缺失 33.00%**  
   这是最关键的发现。三个指标缺失比例**完全一致 (均为 33%)**，说明它们来自同一张底层资产负债表，缺失原因相同。可能原因包括：  
   - **金融行业公司**：其资产负债表格式与一般企业不同，原始表中这些标准科目代码可能不适用，导致字段为空。  
   - **北交所/新三板早期公司**：部分公司上市年份较晚（如 2020 年后），或被归类为“京市”，在数据提取时可能未完全匹配。  
   - **跨表查询生成逻辑**：CSMAR 的“跨表查询”功能在拼表时可能只保留了部分市场和行业的完整财务数据，其余公司因未被纳入查询范围而缺失。

**处理策略：**  
- 后续合并时，财务变量缺失的观测将自然被排除（如计算 ROA 等指标时无法计算）。  
- 若需要更完整的财务数据，可回到 CSMAR 数据库单独下载完整的资产负债表（FS_Combas），与当前数据合并补全。

### 2.1.7 数值列类型转换

将金额、比例、持股数等列强制转换为数值类型（浮点数），无法转换的设为 NaN。

In [11]:
# 常用变量表中需要数值化的列（根据实际字段名调整）
num_cols_av = ["Ysmvosd", "Ysmvttl", "Yretwd", "OwnershipProportion", "ControlProportion",
               "Shrcr1", "Shrhfd5", "Shrz", "FundHoldProportion", "QFIIHoldProportion",
               "BrokerHoldProportion", "BankHoldProportion", "NonFinanceHoldProportion",
               "InsInvestorProp", "StaffNumber", "Boardsize2", "ExecutivesNumber",
               "IndDirector", "SumSalary", "TOP3SumSalary", "DirectorHoldshares", "ManageHoldshares"]

for col in num_cols_av:
    if col in df_av.columns:
        df_av[col] = pd.to_numeric(df_av[col], errors='coerce')

# 跨表查询中所有 FS_Combas- 开头的列都是数值
fs_cols = [c for c in df_ct.columns if c.startswith("FS_Combas-")]
for col in fs_cols:
    df_ct[col] = pd.to_numeric(df_ct[col], errors='coerce')

print("数值类型转换完成")

数值类型转换完成


### 2.1.8 异常值处理

检查并处理以下常见异常：
- 总资产、净资产为负的公司-年度（标记或剔除）
- 比例超出 0~1 范围（如持股比例）的极端值
- 资产负债率 > 1 或 < 0 的异常记录

处理方式：异常值直接设为 NaN，后续分析可排除。

In [12]:
# 总资产、净资产为负检查
if "FS_Combas-A001000000" in df_ct.columns:
    neg_asset = df_ct["FS_Combas-A001000000"] < 0
    if neg_asset.any():
        print(f"总资产为负的记录数: {neg_asset.sum()}，将置为 NaN")
        df_ct.loc[neg_asset, "FS_Combas-A001000000"] = np.nan

if "FS_Combas-A003000000" in df_ct.columns:
    neg_equity = df_ct["FS_Combas-A003000000"] < 0
    if neg_equity.any():
        print(f"净资产为负的记录数: {neg_equity.sum()}，将置为 NaN")
        df_ct.loc[neg_equity, "FS_Combas-A003000000"] = np.nan

# 持股比例限制 0~100（注意原始数据可能是百分数还是小数？）
# 根据 CSMAR，Shrcr1 是百分比数值，如 30 表示 30%，将其规范为 0-100
if "Shrcr1" in df_av.columns:
    # 先检查是否有大于100的异常值
    abnormal = (df_av["Shrcr1"] > 100) | (df_av["Shrcr1"] < 0)
    if abnormal.any():
        print(f"Shrcr1 异常（超出0~100）记录数: {abnormal.sum()}，将置为 NaN")
        df_av.loc[abnormal, "Shrcr1"] = np.nan

# 类似处理其他比例变量...

print("异常值处理完成")

净资产为负的记录数: 368，将置为 NaN
异常值处理完成


### 2.1.9 保存清洗后的数据

将清洗后的常用变量表和跨表查询表分别保存为 `data/clean/firm_year_clean.csv`（此处先保存主表，待合并后再生成最终版）。
暂存两张清洗表，为后续合并做准备。

In [13]:
clean_dir = Path("data/clean")
clean_dir.mkdir(parents=True, exist_ok=True)

# 常用变量表（已清洗）
df_av.to_csv(clean_dir / "annual_vars_clean.csv", index=False, encoding="utf-8-sig")
# 跨表查询表
df_ct.to_csv(clean_dir / "cross_table_clean.csv", index=False, encoding="utf-8-sig")

print("清洗后单表已保存")

清洗后单表已保存


## 2.2样本口径说明

### 公司范围
- **A股上市公司**：包含上海证券交易所、深圳证券交易所及北京证券交易所上市的公司。
- 不包含B股、H股及其他境外上市股票，除非特别说明。

### 时间范围
- **2000年至最新可得年份**：起始年份为2000年，截止年份为数据中最近完整的会计年度。
- 若某个指标的实际可得年份晚于2000年（如公司治理数据从2003年起），则在分析时注明实际起始年份。

### 频率
- **年度数据**：每个公司每个会计年度一条观测，使用年末（12月31日）数据或年度汇总值。

### 报表口径
- **优先使用合并报表**：当原始数据包含母公司报表与合并报表时，选择合并报表数据（通常以字段 `Typrep` 标识，合并报表取值为 `'A'`）。
- 若原始数据未区分报表类型，默认所有记录均为合并报表口径，并在报告中说明。

### 行业分类
- 使用 **CSMAR 行业代码**（如 `IndustryCode` 字段）。
- 至少保留**行业大类代码**（即行业代码的第一个字母），常见大类包括：
  - **C**：制造业
  - **D**：电力、热力、燃气及水生产和供应业
  - **E**：建筑业
  - **F**：批发和零售业
  - **G**：交通运输、仓储和邮政业
  - **J**：金融业
  - **K**：房地产业
- 若分析需要，可进一步细分至二级行业。

### 金融业公司处理
- **保留金融业公司**（行业代码 `J`），不将其从样本中剔除。
- **重要说明**：金融业（银行、保险、证券等）的资产负债结构与实体企业显著不同，其资产负债率通常远高于非金融公司。因此，在分析资产负债率等指标时，**需分组讨论**或明确注明“包含/不包含金融业”的结果对比。

In [14]:
# ==================== 样本口径实施 ====================
# 1. 报表类型筛选
if 'Typrep' in df_ct.columns:
    print("报表类型分布:", df_ct['Typrep'].value_counts().to_dict())
    df_ct = df_ct[df_ct['Typrep'] == 'A'].copy()
    print("筛选合并报表后样本量:", len(df_ct))
else:
    print("无 Typrep 字段，默认所有数据为合并报表口径。")

# 2. 行业分类（尝试多个可能的列名）
for col in ['IndustryCode', 'IndustryCodeC', 'IndustryCodeD']:
    if col in df_info.columns:
        df_info['industry_major'] = df_info[col].astype(str).str[0]
        break
if 'industry_major' in df_info.columns:
    print("行业大类分布:", df_info['industry_major'].value_counts().to_dict())
    df_info['is_finance'] = (df_info['industry_major'] == 'J')
    print("金融业公司数量:", df_info['is_finance'].sum())
else:
    print("未找到行业代码列，无法提取大类。")

# ==================== 合并数据 ====================
# 统一 df_info 股票代码
df_info['code'] = df_info['Symbol'].astype(str).str.strip().str.zfill(6)

# 合并常用变量和财务数据
df_merged = df_av.merge(df_ct, on=['code', 'year'], how='left', suffixes=('', '_fin'))

# 合并公司信息（行业、上市日期等）
info_cols = ['code', 'industry_major', 'is_finance', 'listing_date'] if 'listing_date' in df_info.columns else ['code', 'industry_major', 'is_finance']
df_merged = df_merged.merge(df_info[info_cols], on='code', how='left')

# ==================== 变量构造 ====================
# 资产负债率
df_merged['lev'] = df_merged['FS_Combas-A002000000'] / df_merged['FS_Combas-A001000000']

# 公司年龄
if 'listing_date' in df_merged.columns:
    df_merged['list_year'] = pd.to_datetime(df_merged['listing_date']).dt.year
    df_merged['age'] = df_merged['year'] - df_merged['list_year']
    df_merged.loc[df_merged['age'] < 0, 'age'] = np.nan

# 董事会独立性
df_merged['board_indep'] = df_merged['IndDirector'] / df_merged['Boardsize2']

# ==================== 保存最终面板 ====================
from pathlib import Path
combined_dir = Path("data/combined")
combined_dir.mkdir(parents=True, exist_ok=True)
df_merged.to_csv(combined_dir / "csmar_firm_year_panel.csv", index=False, encoding="utf-8-sig")
print("最终面板数据已保存，形状:", df_merged.shape)

无 Typrep 字段，默认所有数据为合并报表口径。
行业大类分布: {'C': 40346, 'I': 3582, 'F': 2770, 'G': 2447, 'K': 2065, 'J': 2027, 'D': 2004, 'M': 1477, 'E': 1456, 'H': 1330, 'B': 1274, 'A': 950, 'L': 738, 'N': 686, 'R': 587, 'S': 243, 'Q': 112, 'P': 69, 'O': 7, '没': 1, '行': 1, 'n': 1}
金融业公司数量: 2027
最终面板数据已保存，形状: (1052971, 75)


In [15]:
# 检查并清理无效的行业大类
valid_majors = ['A','B','C','D','E','F','G','H','I','J','K','L','M','N','O','P','Q','R','S']
df_merged['industry_major'] = df_merged['industry_major'].apply(lambda x: x if x in valid_majors else np.nan)
print("清理后行业大类分布:", df_merged['industry_major'].value_counts().to_dict())
# 重新保存面板
df_merged.to_csv("data/combined/csmar_firm_year_panel.csv", index=False, encoding="utf-8-sig")

清理后行业大类分布: {'C': 625397, 'F': 54415, 'G': 46505, 'K': 45890, 'I': 44275, 'D': 41243, 'J': 37834, 'H': 29864, 'B': 24172, 'E': 23736, 'M': 23590, 'A': 17696, 'L': 12075, 'R': 8812, 'N': 8781, 'S': 5757, 'Q': 1772, 'P': 1091, 'O': 42}


### 2.3 指标构建

根据作业要求，构建以下公司-年度财务指标。所有比率型指标均采用小数形式（如0.35表示35%）。

**指标定义与计算方式：**

| 指标 | 名称 | 计算公式 | 说明 |
|------|------|----------|------|
| **Lev** | 总负债率 | 总负债 / 总资产 | 衡量总资产中有多大比例由负债融资 |
| **SL** | 短期负债率 | 流动负债 / 总资产 | 反映短期债务融资的依赖程度 |
| **LL** | 长期负债率 | 非流动负债 / 总资产 | 反映长期债务融资的依赖程度 |
| **SDR** | 短期债务占比 | 流动负债 / 总负债 | 衡量总负债中短期债务的集中度 |
| **Cash** | 现金比率 | 货币资金 / 总资产 | 衡量公司资产中现金及等价物的占比 |
| **TA** | 总资产 | 总资产（直接取值） | 公司全部资产的账面价值，单位元 |
| **TE** | 净资产 | 所有者权益合计（直接取值） | 股东权益总额，单位元 |
| **SLoan** | 短期借款比率 | 短期借款 / 总资产 | 衡量短期银行借款对资产的占比 |
| **LLoan** | 长期借款比率 | 长期借款 / 总资产 | 衡量长期银行借款对资产的占比 |
| **Top1** | 第一大股东持股比例 | 第一大股东持股比例 / 100 | 反映股权集中度 |
| **HHI5** | 前五大股东持股集中度 | (前五大持股比例合计/100)² | 近似值；准确计算需各股东单独持股比例 |
| **Size** | 公司规模 | ln(总资产) | 对总资产取自然对数 |
| **Age** | 上市年龄 | 会计年度 - 上市年份 + 1 | 上市不足一年的按实际年份计算 |

> **注：**
> - 作业原文中的“流动资金率”、“长期持有率”、“全部短债”等表述已根据指标实际计算逻辑（均使用负债类科目）修正为上述标准财务指标。
> - 因当前数据缺少净利润，无法计算标准的总资产收益率（ROA）和净资产收益率（ROE）。故改用总资产（TA）和净资产（TE）作为规模指标，反映公司体量。
> - 比例变量限制在0~1之间，异常值将被缩尾处理。
> - HHI5 因缺少每位大股东单独持股比例，采用前五大持股比例合计的平方作为近似值，正式分析时需注意其局限性。

In [17]:
# ==================== 2.3 指标构建 ====================

# 1. 资产负债相关指标
df_merged['lev'] = df_merged['FS_Combas-A002000000'] / df_merged['FS_Combas-A001000000']    # 总负债率
df_merged['sl']  = df_merged['FS_Combas-A002100000'] / df_merged['FS_Combas-A001000000']    # 短期负债率
df_merged['ll']  = df_merged['FS_Combas-A002200000'] / df_merged['FS_Combas-A001000000']    # 长期负债率
df_merged['sdr'] = df_merged['FS_Combas-A002100000'] / df_merged['FS_Combas-A002000000']    # 短期债务占比

# 2. 现金比率
df_merged['cash'] = df_merged['FS_Combas-A001107000'] / df_merged['FS_Combas-A001000000']

# 3. 规模变量（替代原本无法计算的 ROA/ROE）
df_merged['ta'] = df_merged['FS_Combas-A001000000']   # 总资产
df_merged['te'] = df_merged['FS_Combas-A003000000']   # 净资产

# 4. 借款比率
df_merged['sloan'] = df_merged['FS_Combas-A002101000'] / df_merged['FS_Combas-A001000000']  # 短期借款/总资产
df_merged['lloan'] = df_merged['FS_Combas-A002201000'] / df_merged['FS_Combas-A001000000']  # 长期借款/总资产

# 5. 股权结构
df_merged['top1'] = df_merged['Shrcr1'] / 100                       # 第一大股东持股比例（小数）
df_merged['hhi5_approx'] = (df_merged['Shrhfd5'] / 100) ** 2       # 前五大持股集中度近似值

# 6. 公司规模
df_merged['size'] = np.log(df_merged['FS_Combas-A001000000'])

# 7. 上市年龄
if 'list_year' not in df_merged.columns:
    df_merged['list_year'] = pd.to_datetime(df_merged['listing_date']).dt.year
df_merged['age'] = df_merged['year'] - df_merged['list_year'] + 1
df_merged.loc[df_merged['age'] < 0, 'age'] = np.nan

# 8. 比例变量缩尾（限制在0~1之间）
ratio_cols = ['lev', 'sl', 'll', 'sdr', 'cash', 'sloan', 'lloan', 'top1', 'hhi5_approx']
for col in ratio_cols:
    df_merged[col] = df_merged[col].clip(0, 1)

# 9. 保存最终面板数据
df_merged.to_csv("data/combined/csmar_firm_year_panel.csv", index=False, encoding="utf-8-sig")

print("✅ 指标构建完成，最终数据形状:", df_merged.shape)
print("新构建的指标列：lev, sl, ll, sdr, cash, ta, te, sloan, lloan, top1, hhi5_approx, size, age")

✅ 指标构建完成，最终数据形状: (1052971, 86)
新构建的指标列：lev, sl, ll, sdr, cash, ta, te, sloan, lloan, top1, hhi5_approx, size, age


### 2.4 离群值处理

对核心比率指标按年度进行缩尾处理（winsorize），以减小极端值对分析的影响。

#### 处理范围
- 需缩尾的指标：Lev, SL, LL, SDR, Cash, SLoan, LLoan, HHI5。
- **TA（总资产）与 TE（净资产）为规模变量**，单位是元，不进行缩尾处理。
- Size（公司规模）、Top1（第一大股东持股比例）和 Age（上市年龄）通常也不需要缩尾，暂不处理。

#### 缩尾方法
- 分年度在 **1% 和 99% 分位数**上进行缩尾。
- 保留原始变量（后缀 `_raw`），缩尾后变量沿用原名称（如 `lev_raw` 和 `lev`）。

#### 效果验证
- 对比缩尾前后各指标的均值、标准差和极值，确认离群值被有效压缩，且整体分布未发生扭曲。

In [18]:
import numpy as np
import pandas as pd
from scipy.stats.mstats import winsorize

# ==================== 2.4 离群值处理 ====================

# 1. 备份原始变量，更名为 _raw
raw_vars = ['lev', 'sl', 'll', 'sdr', 'cash', 'sloan', 'lloan', 'hhi5_approx']
for v in raw_vars:
    if v in df_merged.columns:
        df_merged[v + '_raw'] = df_merged[v]

# 2. 定义分年度缩尾函数
def winsorize_series(series, limits=(0.01, 0.01)):
    """对 pandas Series 进行缩尾，limits 为左右尾部比例"""
    return winsorize(series, limits=limits)

# 3. 按年份分组，对指定变量进行缩尾
winsor_cols = ['lev', 'sl', 'll', 'sdr', 'cash', 'sloan', 'lloan', 'hhi5_approx']
for col in winsor_cols:
    if col in df_merged.columns:
        df_merged[col] = df_merged.groupby('year')[col].transform(
            lambda x: pd.Series(winsorize_series(x.values, limits=(0.01, 0.01)), index=x.index)
        )

# 4. 重新 clip 确保仍在 0~1 之间（缩尾后一般不会超出，但安全起见）
for col in winsor_cols:
    df_merged[col] = df_merged[col].clip(0, 1)

# 5. 保存处理后的数据（覆盖或新版本）
df_merged.to_csv("data/combined/csmar_firm_year_panel.csv", index=False, encoding="utf-8-sig")
print("✅ 离群值处理完成，已更新最终面板数据。")

✅ 离群值处理完成，已更新最终面板数据。


In [19]:
# 对比缩尾前后的关键统计量
compare_vars = ['lev', 'sl', 'll', 'sdr', 'cash', 'sloan', 'lloan', 'hhi5_approx']
comparison = []
for v in compare_vars:
    raw = v + '_raw'
    if raw in df_merged.columns:
        raw_data = df_merged[raw].dropna()
        win_data = df_merged[v].dropna()
        comparison.append({
            'Variable': v,
            'Raw Mean': raw_data.mean(),
            'Winsorized Mean': win_data.mean(),
            'Raw Std': raw_data.std(),
            'Winsorized Std': win_data.std(),
            'Raw Min': raw_data.min(),
            'Winsorized Min': win_data.min(),
            'Raw Max': raw_data.max(),
            'Winsorized Max': win_data.max()
        })

comp_df = pd.DataFrame(comparison)
comp_df.round(4)

,Variable,Raw Mean,Winsorized Mean,Raw Std,Winsorized Std,Raw Min,Winsorized Min,Raw Max,Winsorized Max
0,lev,0.4646,0.4658,0.2226,0.2231,0.0,0.0389,1.0000,1.0000
1,sl,0.3601,0.3602,0.1922,0.1918,0.0,0.0000,1.0000,1.0000
2,ll,0.0989,0.0987,0.1144,0.1132,0.0,0.0000,1.0000,1.0000
3,sdr,0.7994,0.7999,0.1918,0.1903,0.0,0.0000,1.0000,1.0000
4,cash,0.0294,0.0293,0.0733,0.0728,0.0,0.0000,0.8450,0.8450
5,sloan,0.1047,0.1045,0.1091,0.1073,0.0,0.0000,1.0000,1.0000
6,lloan,0.0671,0.0672,0.0906,0.0900,0.0,0.0000,0.8460,0.8460
7,hhi5_approx,0.0000,0.0000,0.0000,0.0000,0.0,0.0000,0.0001,0.0001


### 2.5 合并与输出

将清洗后的常用变量表（`annual_vars_clean.csv`）、跨表查询表（`cross_table_clean.csv`）以及公司基本信息表合并为 **公司—年度面板数据**。

#### 合并规则
- **主键**：`code`‑`year`
- **合并方式**：以常用变量表为左表，左连接跨表查询表（财务数据）和公司基本信息。
- **记录行数变化**：在每一步合并后输出样本量，确保数据可追溯。

#### 输出文件
| 文件路径 | 说明 |
|----------|------|
| `data/clean/firm_year_clean.csv` | 清洗后的公司-年度表（不含构造指标，仅标准化后的字段） |
| `data/combined/csmar_firm_year_panel.csv` | 最终面板数据（含所有构造指标及缩尾变量） |
| `output/tables/missing_summary.csv` | 核心变量缺失统计 |
| `output/tables/winsor_summary.csv` | 缩尾前后对比统计 |
| `process_log.txt` | 数据处理关键步骤日志 |

In [20]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# ==================== 2.5 合并与输出 ====================

# 日志记录函数
def log(message, log_file="process_log.txt"):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_line = f"[{timestamp}] {message}"
    print(log_line)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(log_line + "\n")

# 重置日志文件
with open("process_log.txt", "w", encoding="utf-8") as f:
    f.write("数据处理日志\n=============\n")

log("开始合并与输出")

# ====================================================
# 1. 加载清洗后的单表
# ====================================================
df_av = pd.read_csv("data/clean/annual_vars_clean.csv")
df_ct = pd.read_csv("data/clean/cross_table_clean.csv")
df_info = pd.read_excel("data/raw/STK_LISTEDCOINFOANL.xlsx")

log(f"读取 annual_vars_clean.csv: {df_av.shape}")
log(f"读取 cross_table_clean.csv: {df_ct.shape}")
log(f"读取 STK_LISTEDCOINFOANL.xlsx: {df_info.shape}")

# ====================================================
# 2. 主键统一（code-year）
# ====================================================
# 2.1 处理常用变量表
df_av["code"] = df_av["code"].astype(str).str.zfill(6)
# 2.2 处理跨表查询表
df_ct["code"] = df_ct["code"].astype(str).str.zfill(6)
# 2.3 处理公司信息表
df_info["code"] = df_info["Symbol"].astype(str).str.strip().str.zfill(6)

# 提取行业大类（使用行业代码第一字母）
for col in ['IndustryCode', 'IndustryCodeC', 'IndustryCodeD']:
    if col in df_info.columns:
        df_info['industry_major'] = df_info[col].astype(str).str[0]
        break
# 标记金融业
df_info['is_finance'] = (df_info['industry_major'] == 'J')

# 只保留需要的公司信息列
info_keep = ['code', 'industry_major', 'is_finance']
if 'LISTINGDATE' in df_info.columns:
    df_info['listing_date'] = pd.to_datetime(df_info['LISTINGDATE'], errors='coerce')
    info_keep.append('listing_date')
elif 'listing_date' in df_info.columns:
    info_keep.append('listing_date')

df_info_select = df_info[info_keep].drop_duplicates(subset='code')
log(f"公司信息表去重后: {df_info_select.shape}")

# ====================================================
# 3. 合并
# ====================================================
# 步骤1：常用变量 + 跨表查询
df_merged_step1 = df_av.merge(df_ct, on=['code','year'], how='left', suffixes=('','_fin'))
log(f"合并常用变量+跨表查询: {df_merged_step1.shape}")

# 步骤2：加上公司基本信息
df_firm_year_clean = df_merged_step1.merge(df_info_select, on='code', how='left')
log(f"合并公司信息后: {df_firm_year_clean.shape}")

# ====================================================
# 4. 保存清洗后的公司-年度表（不含构造指标）
# ====================================================
clean_dir = Path("data/clean")
clean_dir.mkdir(parents=True, exist_ok=True)
df_firm_year_clean.to_csv(clean_dir / "firm_year_clean.csv", index=False, encoding="utf-8-sig")
log("保存 firm_year_clean.csv 成功")

# ====================================================
# 5. 构建指标（若尚未构建，这里可再次执行；但通常在前面单元格已完成）
# ====================================================
# 如果 df_merged 不存在，说明需要重新构建；此处为安全起见，直接基于 df_firm_year_clean 构建
df = df_firm_year_clean.copy()

# 指标构建（与前面一致，保证变量存在）
df['lev'] = df['FS_Combas-A002000000'] / df['FS_Combas-A001000000']
df['sl']  = df['FS_Combas-A002100000'] / df['FS_Combas-A001000000']
df['ll']  = df['FS_Combas-A002200000'] / df['FS_Combas-A001000000']
df['sdr'] = df['FS_Combas-A002100000'] / df['FS_Combas-A002000000']
df['cash'] = df['FS_Combas-A001107000'] / df['FS_Combas-A001000000']
df['ta'] = df['FS_Combas-A001000000']
df['te'] = df['FS_Combas-A003000000']
df['sloan'] = df['FS_Combas-A002101000'] / df['FS_Combas-A001000000']
df['lloan'] = df['FS_Combas-A002201000'] / df['FS_Combas-A001000000']
df['top1'] = df['Shrcr1'] / 100
df['hhi5_approx'] = (df['Shrhfd5'] / 100) ** 2
df['size'] = np.log(df['FS_Combas-A001000000'])
if 'listing_date' in df.columns:
    df['list_year'] = pd.to_datetime(df['listing_date']).dt.year
    df['age'] = df['year'] - df['list_year'] + 1
    df.loc[df['age'] < 0, 'age'] = np.nan

# 缩尾处理（若尚未缩尾，这里执行；如果前面已做，则跳过）
from scipy.stats.mstats import winsorize
winsor_vars = ['lev','sl','ll','sdr','cash','sloan','lloan','hhi5_approx']
for v in winsor_vars:
    df[v + '_raw'] = df[v]
    df[v] = df.groupby('year')[v].transform(lambda x: pd.Series(winsorize(x.values, limits=(0.01,0.01)), index=x.index))
    df[v] = df[v].clip(0,1)

log("指标构建与缩尾处理完成")

# ====================================================
# 6. 保存最终面板数据
# ====================================================
combined_dir = Path("data/combined")
combined_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(combined_dir / "csmar_firm_year_panel.csv", index=False, encoding="utf-8-sig")
log("保存 csmar_firm_year_panel.csv 成功")

# ====================================================
# 7. 缺失值摘要表
# ====================================================
missing_vars = ['lev','sl','ll','sdr','cash','sloan','lloan','hhi5_approx','top1','size','age']
missing_stats = df[missing_vars].isnull().mean().reset_index()
missing_stats.columns = ['variable','missing_ratio']
missing_stats['missing_count'] = df[missing_vars].isnull().sum().values
output_dir = Path("output/tables")
output_dir.mkdir(parents=True, exist_ok=True)
missing_stats.to_csv(output_dir / "missing_summary.csv", index=False, encoding="utf-8-sig")
log("保存 missing_summary.csv 成功")

# ====================================================
# 8. 缩尾效果摘要表
# ====================================================
winsor_comparison = []
for v in winsor_vars:
    raw_col = v + '_raw'
    if raw_col in df.columns:
        raw = df[raw_col].dropna()
        win = df[v].dropna()
        winsor_comparison.append({
            'variable': v,
            'raw_mean': raw.mean(),
            'winsorized_mean': win.mean(),
            'raw_std': raw.std(),
            'winsorized_std': win.std(),
            'raw_min': raw.min(),
            'winsorized_min': win.min(),
            'raw_max': raw.max(),
            'winsorized_max': win.max()
        })
winsor_summary = pd.DataFrame(winsor_comparison)
winsor_summary.to_csv(output_dir / "winsor_summary.csv", index=False, encoding="utf-8-sig")
log("保存 winsor_summary.csv 成功")

log("所有输出文件已生成，合并与输出步骤结束。")

[2026-05-22 17:27:08] 开始合并与输出
[2026-05-22 17:28:50] 读取 annual_vars_clean.csv: (61456, 36)
[2026-05-22 17:28:50] 读取 cross_table_clean.csv: (81662, 34)
[2026-05-22 17:28:50] 读取 STK_LISTEDCOINFOANL.xlsx: (64173, 40)
[2026-05-22 17:28:51] 公司信息表去重后: (5615, 4)
[2026-05-22 17:28:51] 合并常用变量+跨表查询: (61456, 68)
[2026-05-22 17:28:52] 合并公司信息后: (61456, 71)
[2026-05-22 17:28:59] 保存 firm_year_clean.csv 成功
[2026-05-22 17:29:02] 指标构建与缩尾处理完成
[2026-05-22 17:29:12] 保存 csmar_firm_year_panel.csv 成功
[2026-05-22 17:29:12] 保存 missing_summary.csv 成功
[2026-05-22 17:29:12] 保存 winsor_summary.csv 成功
[2026-05-22 17:29:12] 所有输出文件已生成，合并与输出步骤结束。


In [21]:
from pathlib import Path
print("data/clean/firm_year_clean.csv:", Path("data/clean/firm_year_clean.csv").exists())
print("data/combined/csmar_firm_year_panel.csv:", Path("data/combined/csmar_firm_year_panel.csv").exists())
print("output/tables/missing_summary.csv:", Path("output/tables/missing_summary.csv").exists())
print("output/tables/winsor_summary.csv:", Path("output/tables/winsor_summary.csv").exists())
print("\nprocess_log.txt 最后20行：")
with open("process_log.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()
    for line in lines[-20:]:
        print(line.rstrip())

data/clean/firm_year_clean.csv: True
data/combined/csmar_firm_year_panel.csv: True
output/tables/missing_summary.csv: True
output/tables/winsor_summary.csv: True

process_log.txt 最后20行：
数据处理日志
[2026-05-22 17:27:08] 开始合并与输出
[2026-05-22 17:28:50] 读取 annual_vars_clean.csv: (61456, 36)
[2026-05-22 17:28:50] 读取 cross_table_clean.csv: (81662, 34)
[2026-05-22 17:28:50] 读取 STK_LISTEDCOINFOANL.xlsx: (64173, 40)
[2026-05-22 17:28:51] 公司信息表去重后: (5615, 4)
[2026-05-22 17:28:51] 合并常用变量+跨表查询: (61456, 68)
[2026-05-22 17:28:52] 合并公司信息后: (61456, 71)
[2026-05-22 17:28:59] 保存 firm_year_clean.csv 成功
[2026-05-22 17:29:02] 指标构建与缩尾处理完成
[2026-05-22 17:29:12] 保存 csmar_firm_year_panel.csv 成功
[2026-05-22 17:29:12] 保存 missing_summary.csv 成功
[2026-05-22 17:29:12] 保存 winsor_summary.csv 成功
[2026-05-22 17:29:12] 所有输出文件已生成，合并与输出步骤结束。


In [22]:
!pip install pyarrow

## 3.2 进阶存储：Parquet 格式

在 CSV 基础上，将最终面板数据额外保存为列式存储格式 **Parquet**，并演示其优势。

**优点**：
- 列式存储，读取指定列时速度更快、内存占用更小。
- 内置数据类型和压缩，文件体积远小于 CSV。
- 适合大规模数据分析和分布式计算。

In [23]:
import pandas as pd
from pathlib import Path

# 读取最终面板 CSV（或直接用已有的 df）
df_panel = pd.read_csv("data/combined/csmar_firm_year_panel.csv")

# 保存为 Parquet
parquet_path = "data/combined/csmar_firm_year_panel.parquet"
df_panel.to_parquet(parquet_path, index=False)
print(f"Parquet 文件已保存至: {parquet_path}")

Parquet 文件已保存至: data/combined/csmar_firm_year_panel.parquet


### Parquet 优势演示

1. **按列读取**：只加载需要分析的几列，加快读取速度并节省内存。
2. **Schema 查看**：内置数据类型信息，避免 CSV 的类型推断错误。
3. **速度与体积对比**：量化 Parquet 相较于 CSV 的性能提升。

In [24]:
import os
import time
import pandas as pd
import pyarrow.parquet as pq

# ================= 1. 列式读取优势 =================
# 只读取 5 个核心列（根据你的实际变量名调整，这里以 Lev 等为例）
columns_to_read = ["code", "year", "lev", "ta", "cash"]
df_sub = pd.read_parquet(parquet_path, columns=columns_to_read)
print(f"只读取 {len(columns_to_read)} 列，数据形状: {df_sub.shape}")
print(df_sub.head())

# ================= 2. 查看 Schema =================
schema = pq.read_schema(parquet_path)
print("\nParquet Schema:")
print(schema)

# ================= 3. 速度与体积对比 =================
csv_path = "data/combined/csmar_firm_year_panel.csv"

# 3.1 体积对比
csv_size_mb = os.path.getsize(csv_path) / 1024 / 1024
parquet_size_mb = os.path.getsize(parquet_path) / 1024 / 1024
print(f"\nCSV 文件大小: {csv_size_mb:.2f} MB")
print(f"Parquet 文件大小: {parquet_size_mb:.2f} MB")
print(f"压缩比: {csv_size_mb / parquet_size_mb:.1f}x")

# 3.2 读取速度对比（全量读取）
# 先预热一下，避免首次读取受缓存影响
pd.read_csv(csv_path)
pd.read_parquet(parquet_path)

t0 = time.time()
pd.read_csv(csv_path)
csv_time = time.time() - t0
print(f"\nCSV 全量读取耗时: {csv_time:.3f} 秒")

t0 = time.time()
pd.read_parquet(parquet_path)
parquet_time = time.time() - t0
print(f"Parquet 全量读取耗时: {parquet_time:.3f} 秒")
print(f"加速比: {csv_time / parquet_time:.1f}x")

只读取 5 列，数据形状: (61456, 5)
   code  year  lev  ta  cash
0     1  2000  NaN NaN   NaN
1     1  2001  NaN NaN   NaN
2     1  2002  NaN NaN   NaN
3     1  2003  NaN NaN   NaN
4     1  2004  NaN NaN   NaN

Parquet Schema:
Stkcd: int64
accper: int64
stknme: string
AnaAttention: double
Audittyp: string
InternationalBig4: double
Ysmvosd: double
Ysmvttl: double
Yretwd: double
PropertyRightsNature: double
Seperation: double
ActualControllerNatureID: string
OwnershipProportion: double
ControlProportion: double
Shrcr1: double
Shrhfd5: double
Shrz: double
FundHoldProportion: double
QFIIHoldProportion: double
BrokerHoldProportion: double
BankHoldProportion: double
NonFinanceHoldProportion: double
InsInvestorProp: double
StaffNumber: double
ConcurrentPosition: double
Boardsize2: double
ExecutivesNumber: double
IndDirector: double
SumSalary: double
TOP3SumSalary: double
Ynshrtrd: int64
DirectorHoldshares: double
ManageHoldshares: double
code: int64
year: int64
accper_dt: string
stknme_fin: string
listi

### 速度与体积差异分析

**本次数据规模下（约 105 万行 × 75 列）：**
- **体积**：Parquet 文件通常只有 CSV 的 1/3 ~ 1/5，因为列式压缩效率高（尤其对重复值多的列如行业代码）。
- **速度**：全量读取时 Parquet 略快于 CSV，但差异不算巨大；**按列读取时优势非常明显**，因为 Parquet 只扫描所需列的数据块，而 CSV 必须解析整行。

**如果扩展到全部 A 股、季度数据或多表合并：**
- **全部 A 股 + 季度频率**：数据量将增长至数百万到千万行。Parquet 的列式存储和压缩优势会更突出，体积可能只有 CSV 的 1/10，读取速度可能快 3~5 倍。
- **多张数据库表合并**：若涉及多个来源（资产负债表、利润表、行情表），Parquet 支持分区和谓词下推，可按年份或股票代码分区存储，查询某只股票所有年份数据时几乎瞬间完成，而 CSV 需要遍历所有文件。
- **长期存档和复用**：Parquet 自带的 Schema 避免了每次读取时重新指定数据类型，降低了出错概率，特别适合团队协作和自动化流水线。